In [0]:
%python
########## we should start from using sklearn ############################
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# 1. Read data
spark_df = spark.table("ai2605.ai.petstreaming_sales")

# 2. Prepare the label (is_subscription -> label)
df_prepared = spark_df.withColumn("label", col("is_subscription").cast("double"))

# 3. Handle Categorical Data: Convert 'category' to indices, then One-Hot encode
# StringIndexer converts string names to numeric indices (e.g., Accessories=0, Food=1)
indexer = StringIndexer(inputCol="category", outputCol="category_idx")

# OneHotEncoder converts indices into a sparse binary vector (e.g., [1,0,0])
encoder = OneHotEncoder(inputCol="category_idx", outputCol="category_vec")

# 4. Assemble all features (numerical + encoded categorical)
assembler = VectorAssembler(
    inputCols=["units_sold", "price", "category_vec"], 
    outputCol="features"
)

# 5. Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label")

# 6. Pipeline
pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(df_prepared)

In [0]:
%python
import mlflow
from mlflow.models.signature import infer_signature
print(mlflow.get_registry_uri())

model_name = "ai2605.ai.petstreaming_sales_model"
volume_path = "/Volumes/ai2605/ai/ml_temp"

with mlflow.start_run(run_name = "ml_0625_01") as run:
    # 1. Train your model (assuming 'model' is your fitted pipeline)
    # model = pipeline.fit(df_prepared)
    signature = infer_signature(df_prepared, model.transform(df_prepared))
    # 2. Log and Register the model
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        registered_model_name= model_name,
        signature = signature,
        dfs_tmpdir=volume_path
    )

    print(f"Model logged and registered as: {model_name}")

